# 09 - K-Means Zone Selection & Sensitivity Analysis

**Title**: `09_kmeans_zone_selection.ipynb`  
**Purpose**: Evaluate spatial clustering of municipal 311 complaints across Brooklyn into operational response zones using K-Means. Evaluates three sample-weighting variants (raw complaint counts, capped coordinate counts at p99, and capped-balanced group weighting) across zone granularities K = 2 to 15. Evaluates clustering quality via weighted inertia (elbow criterion) and weighted silhouette scores, assesses multi-seed cluster assignment stability (Adjusted Rand Index), examines group concentration and lift profiles at K = 8, and measures the geographic pulling effect of the single largest address hotspot (1869 83 STREET). Evidence is presented without selecting a final K or variant.  
**Inputs**: `data/processed/complaints_clean.csv`, `data/processed/projection.json`  
**Outputs**: Weighted inertia and silhouette evaluation tables, elbow and silhouette diagnostic plots for each weighting variant (`report/figures/09_*.png`), multi-seed ARI stability matrix across K in {4, 6, 8, 10, 12}, K=8 zone concentration and dominant-group lift tables, and spatial displacement diagnostics for the 1869 83 STREET coordinate outlier.  
**Runtime**: ~4-6 minutes

In [1]:
import sys; from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "environment.yml").exists()); sys.path.insert(0, str(ROOT))
from src.nbutils import Checker, md
RUN_HEAVY = False
RUN_API = False

In [2]:
import numpy as np
import pandas as pd
from src.clustering.kmeans_zones import (
    distinct_locations,
    build_weights,
    fit_kmeans,
    weighted_silhouette_sample,
    zone_table,
    evaluate_k_range,
    evaluate_stability,
    plot_elbow,
    plot_silhouette,
    GROUPS,
)
from src.preprocessing.features import load_projection_constants, unproject

clean_path = ROOT / "data/processed/complaints_clean.csv"
proj_path = ROOT / "data/processed/projection.json"

print(f"Loading cleaned complaints from {clean_path.relative_to(ROOT)}...")
df = pd.read_csv(
    clean_path,
    dtype={"id": str, "resolution_id": str},
    keep_default_na=False,
)
lat0, lon0 = load_projection_constants(proj_path)
print(f"Loaded {len(df):,} complaints. Fixed projection origin: lat0={lat0}, lon0={lon0}")

print("Extracting distinct spatial coordinates per complaint group...")
locs = distinct_locations(df)
print(f"Extracted {len(locs):,} distinct (group, x_m, y_m) locations.")

Loading cleaned complaints from data\processed\complaints_clean.csv...


Loaded 236,507 complaints. Fixed projection origin: lat0=40.661641, lon0=-73.952455
Extracting distinct spatial coordinates per complaint group...


Extracted 60,729 distinct (group, x_m, y_m) locations.


In [3]:
print("=== Section 1: Sample Weighting Construction ===")
w_raw = build_weights(locs, "raw")
w_capped = build_weights(locs, "capped")
w_balanced = build_weights(locs, "capped_balanced")

print(f"Raw variant total weight: {w_raw.sum():,.1f}")
print(f"Capped variant total weight: {w_capped.sum():,.1f}")
print(f"Capped-Balanced variant total weight: {w_balanced.sum():,.1f}")

weight_summary = []
for g in GROUPS:
    idx = locs["group"] == g
    weight_summary.append({
        "group": g,
        "distinct_locs": int(idx.sum()),
        "raw_weight": float(w_raw[idx].sum()),
        "capped_weight": float(w_capped[idx].sum()),
        "capped_balanced_weight": float(w_balanced[idx].sum()),
    })
df_weights = pd.DataFrame(weight_summary)
print("\n--- Weight Distribution Across Groups ---")
print(df_weights.to_string(index=False))

=== Section 1: Sample Weighting Construction ===


Raw variant total weight: 236,507.0
Capped variant total weight: 189,264.0
Capped-Balanced variant total weight: 189,264.0

--- Weight Distribution Across Groups ---
       group  distinct_locs  raw_weight  capped_weight  capped_balanced_weight
       flood           3594      5520.0         5151.0                 47316.0
       noise          46484    211843.0       165936.0                 47316.0
        road           8378     16138.0        15227.0                 47316.0
sewer_backup           2273      3006.0         2950.0                 47316.0


In [4]:
print("=== Section 2: K-Means Evaluation (K = 2..15, seed = 42) ===")
metrics_list = []
for v in ["raw", "capped", "capped_balanced"]:
    print(f"Evaluating variant: {v}...")
    df_v = evaluate_k_range(locs, v, k_min=2, k_max=15, seed=42, n_samples=10000)
    metrics_list.append(df_v)

df_k_metrics = pd.concat(metrics_list, ignore_index=True)

print("\n--- Variant x K Evaluation Table (First 15 rows) ---")
print(df_k_metrics.head(15).to_string(index=False))

print("\n--- Full Metrics Table: Variant x K x Inertia x Silhouette ---")
for v in ["raw", "capped", "capped_balanced"]:
    print(f"\nVariant: {v}")
    print(df_k_metrics[df_k_metrics["variant"] == v][["K", "inertia", "silhouette"]].to_string(index=False))

=== Section 2: K-Means Evaluation (K = 2..15, seed = 42) ===
Evaluating variant: raw...


Evaluating variant: capped...


Evaluating variant: capped_balanced...



--- Variant x K Evaluation Table (First 15 rows) ---
variant  K      inertia  silhouette
    raw  2 2.882776e+12    0.454803
    raw  3 1.999999e+12    0.389557
    raw  4 1.499073e+12    0.383232
    raw  5 1.220065e+12    0.397178
    raw  6 9.483629e+11    0.397047
    raw  7 8.041177e+11    0.398966
    raw  8 6.962084e+11    0.405918
    raw  9 6.085791e+11    0.409673
    raw 10 5.286355e+11    0.415181
    raw 11 4.672136e+11    0.429536
    raw 12 4.200745e+11    0.431690
    raw 13 3.855512e+11    0.433143
    raw 14 3.561247e+11    0.426927
    raw 15 3.303415e+11    0.432639
 capped  2 2.338756e+12    0.442649

--- Full Metrics Table: Variant x K x Inertia x Silhouette ---

Variant: raw
 K      inertia  silhouette
 2 2.882776e+12    0.454803
 3 1.999999e+12    0.389557
 4 1.499073e+12    0.383232
 5 1.220065e+12    0.397178
 6 9.483629e+11    0.397047
 7 8.041177e+11    0.398966
 8 6.962084e+11    0.405918
 9 6.085791e+11    0.409673
10 5.286355e+11    0.415181
11 4.672136e

In [5]:
print("=== Section 3: Diagnostic Visualizations ===")
figures_dir = ROOT / "report/figures"

for v in ["raw", "capped", "capped_balanced"]:
    elbow_path = figures_dir / f"09_elbow_{v}.png"
    sil_path = figures_dir / f"09_silhouette_{v}.png"

    plot_elbow(df_k_metrics, v, elbow_path)
    print(f"Saved elbow plot: {elbow_path.relative_to(ROOT)}")

    plot_silhouette(df_k_metrics, v, sil_path)
    print(f"Saved silhouette plot: {sil_path.relative_to(ROOT)}")

=== Section 3: Diagnostic Visualizations ===


Saved elbow plot: report\figures\09_elbow_raw.png


Saved silhouette plot: report\figures\09_silhouette_raw.png


Saved elbow plot: report\figures\09_elbow_capped.png


Saved silhouette plot: report\figures\09_silhouette_capped.png


Saved elbow plot: report\figures\09_elbow_capped_balanced.png


Saved silhouette plot: report\figures\09_silhouette_capped_balanced.png


In [6]:
print("=== Section 4: Cluster Assignment Stability (Mean Pairwise ARI) ===")
df_stability = evaluate_stability(
    locs,
    variants=("raw", "capped", "capped_balanced"),
    k_values=(4, 6, 8, 10, 12),
    seeds=(0, 1, 2, 3, 4),
)
print("\n--- Mean Pairwise Adjusted Rand Index (ARI) Across 5 Seeds (0 to 4) ---")
print(df_stability.to_string(index=False))

=== Section 4: Cluster Assignment Stability (Mean Pairwise ARI) ===



--- Mean Pairwise Adjusted Rand Index (ARI) Across 5 Seeds (0 to 4) ---
 K      raw   capped  capped_balanced
 4 0.973754 0.976771         0.991846
 6 0.982721 0.989012         0.993173
 8 0.973956 0.981974         0.991112
10 0.942133 0.996055         0.966975
12 0.995065 0.930409         0.957474


In [7]:
print("=== Section 5: Zone Concentration & Lift Profiles at K = 8 ===")
X = locs[["x_m", "y_m"]].values
city_counts = {g: locs.loc[locs["group"] == g, "count"].sum() for g in GROUPS}

zone_tables = {}
concentration_summary = []

for v in ["raw", "capped", "capped_balanced"]:
    w = build_weights(locs, v)
    km8 = fit_kmeans(X, w, k=8, seed=42)
    zt = zone_table(locs, km8.labels_, km8.cluster_centers_, lat0=lat0, lon0=lon0)
    zone_tables[v] = zt

    noise_dom_count = int((zt["dominant_group"] == "noise").sum())

    print(f"\n--- Zone Table: Variant '{v}' (K=8, seed=42) ---")
    cols_to_print = [
        "zone",
        "centre_lat",
        "centre_lon",
        "total_count",
        "dominant_group",
        "lift_noise",
        "lift_road",
        "lift_flood",
        "lift_sewer_backup",
    ]
    print(zt[cols_to_print].to_string(index=False))
    print(f"Number of zones whose dominant_group (by lift) is noise: {noise_dom_count}")

    for g in GROUPS:
        g_shares = [zt.loc[z, f"count_{g}"] / city_counts[g] for z in range(8)]
        max_sh = float(max(g_shares))
        max_z = int(np.argmax(g_shares))
        concentration_summary.append({
            "variant": v,
            "group": g,
            "noise_dominant_zones": noise_dom_count,
            "max_single_zone_share": max_sh,
            "max_share_zone": max_z,
        })

df_conc = pd.DataFrame(concentration_summary)
print("\n--- Max Share of Any Single Zone in Group Rows (K=8) ---")
print(df_conc.to_string(index=False))

=== Section 5: Zone Concentration & Lift Profiles at K = 8 ===



--- Zone Table: Variant 'raw' (K=8, seed=42) ---
 zone  centre_lat  centre_lon  total_count dominant_group  lift_noise  lift_road  lift_flood  lift_sewer_backup
    0   40.651237  -73.948096        45044          noise    1.026976   0.709924    0.838000           0.953698
    1   40.601039  -73.993743        22137          noise    1.014401   0.857987    0.930960           0.874322
    2   40.689175  -73.927313        42081          noise    1.029141   0.761653    0.673008           0.826402
    3   40.637641  -74.008152        19261           road    0.937900   1.718825    1.267947           1.025298
    4   40.714743  -73.952905        24880          noise    1.028341   0.762216    0.817990           0.613488
    5   40.602226  -73.950035        19746          flood    0.904234   1.557115    2.501814           2.000228
    6   40.684316  -73.977581        37464          noise    1.010249   0.987346    0.855446           0.611130
    7   40.665954  -73.889140        25894   sewer_bac


--- Zone Table: Variant 'capped' (K=8, seed=42) ---
 zone  centre_lat  centre_lon  total_count dominant_group  lift_noise  lift_road  lift_flood  lift_sewer_backup
    0   40.666793  -73.889362        25002   sewer_backup    0.972552   1.284873    0.915106           1.560853
    1   40.632785  -73.946312        24415   sewer_backup    0.940971   1.364985    1.737334           1.846515
    2   40.715289  -73.952955        24518          noise    1.027860   0.763906    0.831815           0.612919
    3   40.663339  -73.944470        38329          noise    1.049462   0.544474    0.566742           0.755397
    4   40.594083  -73.970203        33215          flood    0.993574   0.924366    1.413778           1.099104
    5   40.682375  -73.979929        35867          noise    1.009505   0.999029    0.842168           0.625180
    6   40.633503  -74.009682        19698           road    0.923044   1.868925    1.374674           1.070453
    7   40.692722  -73.926072        35463         


--- Zone Table: Variant 'capped_balanced' (K=8, seed=42) ---
 zone  centre_lat  centre_lon  total_count dominant_group  lift_noise  lift_road  lift_flood  lift_sewer_backup
    0   40.680038  -73.979533        39475          noise    1.014470   0.951155    0.819464           0.574018
    1   40.632604  -73.917194        12741   sewer_backup    0.869061   1.820840    2.367413           3.309911
    2   40.630535  -74.006939        28487           road    0.983765   1.277907    0.950551           0.742952
    3   40.595699  -73.967012        25320          flood    0.944332   1.301727    1.959520           1.541250
    4   40.713236  -73.950159        29760          noise    1.030968   0.762804    0.748644           0.552546
    5   40.670215  -73.884358        20164           road    0.985648   1.247923    0.709700           1.213497
    6   40.683859  -73.926347        46451          noise    1.027883   0.744895    0.694552           0.965461
    7   40.645276  -73.956426        34109

In [8]:
print("=== Section 6: Hotspot Pull Diagnostic (1869 83 STREET) ===")
addr_match = df[df["incident_address"] == "1869 83 STREET"]
addr_count = len(addr_match)
addr_x = float(addr_match["x_m"].iloc[0])
addr_y = float(addr_match["y_m"].iloc[0])
addr_lat = float(addr_match["lat"].iloc[0])
addr_lon = float(addr_match["lon"].iloc[0])

# For raw variant at K=8
zt_raw = zone_tables["raw"]
km8_raw_centers = zt_raw[["centre_x_m", "centre_y_m"]].values
dists_m = np.sqrt(np.sum((km8_raw_centers - np.array([addr_x, addr_y])) ** 2, axis=1))
nearest_zone = int(np.argmin(dists_m))
min_dist_m = float(dists_m[nearest_zone])
nearest_zone_size = int(zt_raw.loc[nearest_zone, "total_count"])
addr_share_in_zone = float(addr_count / nearest_zone_size * 100.0)

print(f"Target Hotspot Address: 1869 83 STREET")
print(f"Total Complaints at Address: {addr_count:,}")
print(f"Address Coordinates: lat={addr_lat:.6f}, lon={addr_lon:.6f} (x_m={addr_x:.2f}, y_m={addr_y:.2f})")
print(f"Nearest Zone Center (Raw Variant, K=8): Zone {nearest_zone}")
print(f"Distance to Nearest Zone Center: {min_dist_m:.2f} meters")
print(f"Nearest Zone Total Size: {nearest_zone_size:,} complaints")
print(f"Address Share in Nearest Zone: {addr_share_in_zone:.2f}%")

=== Section 6: Hotspot Pull Diagnostic (1869 83 STREET) ===


Target Hotspot Address: 1869 83 STREET
Total Complaints at Address: 8,844
Address Coordinates: lat=40.608101, lon=-73.999384 (x_m=-3962.89, y_m=-5918.30)
Nearest Zone Center (Raw Variant, K=8): Zone 1
Distance to Nearest Zone Center: 914.45 meters
Nearest Zone Total Size: 22,137 complaints
Address Share in Nearest Zone: 39.95%


In [9]:
print("=== Section 7: Verification Checks ===")
checker = Checker()

# 1. Total rows in complaints_clean.csv == 236507
checker.check("Total rows in complaints_clean.csv = 236507", len(df), 236507)

# 2. Distinct (group, x_m, y_m) location count equals the distinct-coordinates count per group summed (compute both ways)
summed_group_coords = sum(
    df[df["group"] == g].groupby(["x_m", "y_m"]).ngroups
    for g in df["group"].unique()
)
checker.check(
    "Distinct (group, x_m, y_m) location count equals distinct coordinates summed per group",
    len(locs),
    summed_group_coords,
)

# 3. Weights of raw variant sum to 236507
checker.check(
    "Weights of raw variant sum to 236507",
    int(round(float(w_raw.sum()))),
    236507,
)

# 4. capped_balanced group totals are equal within 1e-6
cb_group_sums = [
    float(w_balanced[locs["group"] == g].sum())
    for g in sorted(locs["group"].unique())
]
max_cb_diff = float(max(cb_group_sums) - min(cb_group_sums))
checker.check(
    "capped_balanced group totals are equal within 1e-6",
    max_cb_diff,
    0.0,
    tol=1e-6,
)

checker.summary()

=== Section 7: Verification Checks ===
[PASS] Total rows in complaints_clean.csv = 236507: actual=236507, expected=236507


[PASS] Distinct (group, x_m, y_m) location count equals distinct coordinates summed per group: actual=60729, expected=60729
[PASS] Weights of raw variant sum to 236507: actual=236507, expected=236507
[PASS] capped_balanced group totals are equal within 1e-6: actual=7.275957614183426e-12, expected=0.0
CHECKS: 4 passed, 0 failed


In [10]:
raw_sub = df_k_metrics[df_k_metrics["variant"] == "raw"].sort_values("silhouette", ascending=False)
raw_sil_max_k = int(raw_sub["K"].iloc[0])
raw_sil_max_val = float(raw_sub["silhouette"].iloc[0])

capped_sub = df_k_metrics[df_k_metrics["variant"] == "capped"].sort_values("silhouette", ascending=False)
capped_sil_max_k = int(capped_sub["K"].iloc[0])
capped_sil_max_val = float(capped_sub["silhouette"].iloc[0])

bal_sub = df_k_metrics[df_k_metrics["variant"] == "capped_balanced"].sort_values("silhouette", ascending=False)
bal_sil_max_k = int(bal_sub["K"].iloc[0])
bal_sil_max_val = float(bal_sub["silhouette"].iloc[0])

findings_09 = f"""### Findings
- **Data Integrity & Spatial Footprint**: Evaluated exactly {len(df):,} cleaned complaints aggregated across {len(locs):,} distinct `(group, x_m, y_m)` coordinates in Brooklyn. The raw weighting sums to exactly {int(round(float(w_raw.sum()))):,} complaint units, while coordinate count capping (p99) yields an effective volume of {float(w_capped.sum()):,.1f} units.
- **Clustering Geometry Spectrum (K = 2 to 15)**:
  - **Raw Variant**: Weighted inertia drops monotonically from {df_k_metrics[(df_k_metrics['variant']=='raw') & (df_k_metrics['K']==2)]['inertia'].iloc[0]:.2e} at K=2 to {df_k_metrics[(df_k_metrics['variant']=='raw') & (df_k_metrics['K']==15)]['inertia'].iloc[0]:.2e} at K=15. Weighted silhouette score achieves its peak value of {raw_sil_max_val:.4f} at K={raw_sil_max_k}.
  - **Capped Variant**: Inertia drops from {df_k_metrics[(df_k_metrics['variant']=='capped') & (df_k_metrics['K']==2)]['inertia'].iloc[0]:.2e} at K=2 to {df_k_metrics[(df_k_metrics['variant']=='capped') & (df_k_metrics['K']==15)]['inertia'].iloc[0]:.2e} at K=15. Silhouette score peaks at {capped_sil_max_val:.4f} at K={capped_sil_max_k}.
  - **Capped-Balanced Variant**: Inertia drops from {df_k_metrics[(df_k_metrics['variant']=='capped_balanced') & (df_k_metrics['K']==2)]['inertia'].iloc[0]:.2e} at K=2 to {df_k_metrics[(df_k_metrics['variant']=='capped_balanced') & (df_k_metrics['K']==15)]['inertia'].iloc[0]:.2e} at K=15. Silhouette score peaks at {bal_sil_max_val:.4f} at K={bal_sil_max_k}.
- **Multi-Seed Stability (ARI across seeds 0 to 4)**:
  - Across K in {{4, 6, 8, 10, 12}}, all three variants demonstrate strong stability with mean pairwise ARI consistently exceeding 0.93.
  - At K=8, mean ARI is {df_stability.loc[df_stability['K']==8, 'raw'].iloc[0]:.4f} (raw), {df_stability.loc[df_stability['K']==8, 'capped'].iloc[0]:.4f} (capped), and {df_stability.loc[df_stability['K']==8, 'capped_balanced'].iloc[0]:.4f} (capped_balanced).
- **Zone Concentration & Dominant Group Lift at K = 8**:
  - In the raw variant, {int((zone_tables['raw']['dominant_group'] == 'noise').sum())} out of 8 zones have noise as their dominant group by lift. The maximum share of any single zone in a group's rows is {df_conc[(df_conc['variant']=='raw') & (df_conc['group']=='flood')]['max_single_zone_share'].iloc[0]:.4f} for flood, {df_conc[(df_conc['variant']=='raw') & (df_conc['group']=='noise')]['max_single_zone_share'].iloc[0]:.4f} for noise, {df_conc[(df_conc['variant']=='raw') & (df_conc['group']=='road')]['max_single_zone_share'].iloc[0]:.4f} for road, and {df_conc[(df_conc['variant']=='raw') & (df_conc['group']=='sewer_backup')]['max_single_zone_share'].iloc[0]:.4f} for sewer backup.
  - In the capped variant, {int((zone_tables['capped']['dominant_group'] == 'noise').sum())} out of 8 zones have noise as their dominant group by lift.
  - In the capped-balanced variant, {int((zone_tables['capped_balanced']['dominant_group'] == 'noise').sum())} out of 8 zones have noise as their dominant group by lift.
- **Single Hotspot Centroid Pull (1869 83 STREET)**:
  - The single address `1869 83 STREET` accounts for {addr_count:,} complaints. Under the raw variant at K=8, it is located {min_dist_m:.2f} meters from the nearest zone center (Zone {nearest_zone}), representing {addr_share_in_zone:.2f}% of that entire zone's {nearest_zone_size:,} complaints.
- **Neutrality**: No definitive choice of K or weighting variant is selected here; the empirical evidence across inertia, silhouette, stability, concentration, and outlier sensitivity is preserved for downstream decision-making.
"""
md(findings_09)
checker.summary()

### Findings
- **Data Integrity & Spatial Footprint**: Evaluated exactly 236,507 cleaned complaints aggregated across 60,729 distinct `(group, x_m, y_m)` coordinates in Brooklyn. The raw weighting sums to exactly 236,507 complaint units, while coordinate count capping (p99) yields an effective volume of 189,264.0 units.
- **Clustering Geometry Spectrum (K = 2 to 15)**:
  - **Raw Variant**: Weighted inertia drops monotonically from 2.88e+12 at K=2 to 3.30e+11 at K=15. Weighted silhouette score achieves its peak value of 0.4548 at K=2.
  - **Capped Variant**: Inertia drops from 2.34e+12 at K=2 to 2.80e+11 at K=15. Silhouette score peaks at 0.4426 at K=2.
  - **Capped-Balanced Variant**: Inertia drops from 2.49e+12 at K=2 to 3.03e+11 at K=15. Silhouette score peaks at 0.4358 at K=2.
- **Multi-Seed Stability (ARI across seeds 0 to 4)**:
  - Across K in {4, 6, 8, 10, 12}, all three variants demonstrate strong stability with mean pairwise ARI consistently exceeding 0.93.
  - At K=8, mean ARI is 0.9740 (raw), 0.9820 (capped), and 0.9911 (capped_balanced).
- **Zone Concentration & Dominant Group Lift at K = 8**:
  - In the raw variant, 5 out of 8 zones have noise as their dominant group by lift. The maximum share of any single zone in a group's rows is 0.2089 for flood, 0.1956 for noise, 0.1564 for road, and 0.1816 for sewer backup.
  - In the capped variant, 4 out of 8 zones have noise as their dominant group by lift.
  - In the capped-balanced variant, 4 out of 8 zones have noise as their dominant group by lift.
- **Single Hotspot Centroid Pull (1869 83 STREET)**:
  - The single address `1869 83 STREET` accounts for 8,844 complaints. Under the raw variant at K=8, it is located 914.45 meters from the nearest zone center (Zone 1), representing 39.95% of that entire zone's 22,137 complaints.
- **Neutrality**: No definitive choice of K or weighting variant is selected here; the empirical evidence across inertia, silhouette, stability, concentration, and outlier sensitivity is preserved for downstream decision-making.


CHECKS: 4 passed, 0 failed


### Limitations

- K-Means optimizes isotropic spherical clusters in local projected coordinates (x_m, y_m), which does not account for physical travel networks, waterways, or administrative district boundaries (e.g. community boards and police precincts).
- The weighted silhouette score is computed on a representative subsample of 10,000 locations to remain computationally tractable within notebook execution constraints, introducing minor sampling variance across runs with different random seeds.
- Equal group weighting in the capped-balanced variant assigns equal composite mass to low-frequency categories (such as flood and sewer backup) as to noise, which may over-index zone boundaries toward hydrological incident locations at the expense of high-volume citizen demand density.